In [95]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches
from src.elo import prepare_matches, update_elo
from src.simulation import predict_match
from src.features import update_team_history, update_h2h
import numpy as np
from collections import Counter
from src.simulation import run_tournament


In [96]:
#Load everything from earlier notebooks
draw_threshold = joblib.load('draw_threshold.joblib')
wc_model = joblib.load('world_cup_model.joblib')
history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
scaler = joblib.load('scaler.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")

Singular run to demonstrate the process

In [97]:
#Group stage simulation

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

squad_values = {
    "Mexico": 191, "South Korea": 139, "South Africa": 50, "Czech Republic": 188,
    "Canada": 196, "Switzerland": 332, "Qatar": 22, "Bosnia and Herzegovina": 151,
    "Brazil": 923, "Morocco": 498, "Haiti": 55, "Scotland": 170,
    "United States": 385, "Paraguay": 153, "Australia": 77, "Turkey": 473,
    "Germany": 947, "Curaçao": 25, "Ivory Coast": 522, "Ecuador": 368,
    "Netherlands": 804, "Japan": 270, "Sweden": 406, "Tunisia": 70,
    "Belgium": 547, "Egypt": 116, "Iran": 32, "New Zealand": 34,
    "Spain": 1220, "Cape Verde": 54, "Saudi Arabia": 40, "Uruguay": 360,
    "France": 1520, "Senegal": 478, "Iraq": 21, "Norway": 589,
    "Argentina": 782, "Algeria": 256, "Austria": 242, "Jordan": 20,
    "Portugal": 1010, "DR Congo": 140, "Uzbekistan": 85, "Colombia": 300,
    "England": 1360, "Croatia": 387, "Ghana": 234, "Panama": 34
}

rows=[]
for group, teams in wc_groups.items():
    for team in teams:
        rows.append({"Group": group,
                     "Team": team,
                     "Points": 0,
                     "GD": 0,
                     'GF': 0,
                     "GA": 0})
        
group_stage_result = pd.DataFrame(rows) #Gives us the starting group stage
for match in group_stage_matches.itertuples(index = False):
    
    if match.home_team not in country_elo or pd.isna(country_elo[match.home_team]):
        country_elo[match.home_team] = 1500.0
    if match.away_team not in country_elo or pd.isna(country_elo[match.away_team]):
        country_elo[match.away_team] = 1500.0
    
    x = predict_match(match.home_team, match.away_team, wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[match.home_team], country_elo[match.away_team])
    #print(f"{match.home_team} vs {match.away_team} ({x.home_score}-{x.away_score}, result: {x.outcome}), Prob: {x.probability}, diff: {x.diff}")
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'Points'] += x.home_points
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'Points'] += x.away_points
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GD'] += x.home_score - x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GD'] += x.away_score - x.home_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GF'] += x.home_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GF'] += x.away_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GA'] += x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GA'] += x.home_score
    
    if x.outcome == "Home Win": S = 1
    elif x.outcome == "Away Win": S = 0
    else: S = 0.5
    new_away, new_home = update_elo(S, match.neutral, match.K_factor,x.home_score, x.away_score, 
                                    country_elo[match.away_team], 
                                    country_elo[match.home_team])
    
    country_elo[match.home_team] = new_home
    country_elo[match.away_team] = new_away
    update_team_history(match, x.home_score, x.away_score, history_dict)
    update_h2h(match, h2h_dict)
    
group_stage_result = group_stage_result.sort_values(['Group', 'Points', 'GD', 'GF'], ascending=[True, False, False, False]).reset_index(drop=True)    
print(group_stage_result) #Predicted group stage results




   Group                    Team  Points  GD  GF  GA
0      A                  Mexico       9   9  10   1
1      A             South Korea       4   0   2   2
2      A            South Africa       4  -3   6   9
3      A          Czech Republic       0  -6   5  11
4      B  Bosnia and Herzegovina       5   2   3   1
5      B                  Canada       3   0   2   2
6      B             Switzerland       3   0   1   1
7      B                   Qatar       2  -2   2   4
8      C                  Brazil       9   4   7   3
9      C                 Morocco       4   2   5   3
10     C                Scotland       4   0   7   7
11     C                   Haiti       0  -6   2   8
12     D                Paraguay       5   1   5   4
13     D               Australia       4   3   5   2
14     D                  Turkey       4  -3   5   8
15     D           United States       3  -1   6   7
16     E                 Ecuador       7   8  11   3
17     E                 Germany       5   3  

In [98]:
#Get teams that move on to round of 32

top2 = group_stage_result.groupby('Group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('Group').nth(2).copy() #All teams that placed third (only 8 of them move on)

best8_third = third.sort_values(
    ['Points', 'GD', 'GF'], 
    ascending=[False, False, False]
).head(8)

thirds_slot_map = { #Needs to be fixed
    'E': 'ABCDEFGHIJKL',
    'I': 'ABCDEFGHIJKL', 
    'A': 'ABCDEFGHIJKL',
    'L': 'ABCDEFGHIJKL',
    'D': 'ABCDEFGHIJKL',
    'G': 'ABCDEFGHIJKL',
    'B': 'ABCDEFGHIJKL',
    'K': 'ABCDEFGHIJKL',
}

winners = {g: teams.iloc[0]['Team'] for g, teams in group_stage_result.groupby('Group')}
runners = {g: teams.iloc[1]['Team'] for g, teams in group_stage_result.groupby('Group')}

available_thirds = {row.Group: row.Team for row in best8_third.itertuples()}
assignments = {}
used = set()
for winner_group, eligible_groups in thirds_slot_map.items():
    for g in eligible_groups:
        if g in available_thirds and g not in used:
            assignments[winner_group] = available_thirds[g]
            used.add(g)
            break

r32_matchups = {
    73: (runners['A'], runners['B']),
    74: (winners['E'], assignments['E']),
    75: (winners['F'], runners['C']),
    76: (winners['C'], runners['F']),
    77: (winners['I'], assignments['I']),
    78: (runners['E'], runners['I']),
    79: (winners['A'], assignments['A']),
    80: (winners['L'], assignments['L']),
    81: (winners['D'], assignments['D']),
    82: (winners['G'], assignments['G']),
    83: (runners['K'], runners['L']),
    84: (winners['H'], runners['J']),
    85: (winners['B'], assignments['B']),
    86: (winners['J'], runners['H']),
    87: (winners['K'], assignments['K']),
    88: (runners['D'], runners['G'])
}

r32_rows = []
r16_teams = {}

for match, teams in r32_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])
    
    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]

    r16_teams[match] = winner
    r32_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[teams[0]], 
                                    country_elo[teams[1]])
    
    country_elo[teams[0]] = new_home
    country_elo[teams[1]] = new_away


South Korea 1 - 0 Canada → South Korea
Ecuador 2 - 2 South Africa → South Africa
Netherlands 2 - 2 Morocco → Netherlands
Brazil 4 - 2 Tunisia → Brazil
Norway 4 - 1 Switzerland → Norway
Germany 1 - 1 Senegal → Germany
Mexico 1 - 0 Scotland → Mexico
England 3 - 3 Turkey → Turkey
Paraguay 1 - 2 Japan → Japan
Belgium 7 - 0 Cape Verde → Belgium
Uzbekistan 0 - 0 Croatia → Croatia
Spain 2 - 0 Austria → Spain
Bosnia and Herzegovina 1 - 1 Algeria → Bosnia and Herzegovina
Argentina 2 - 3 Uruguay → Uruguay
Colombia 1 - 1 Panama → Panama
Australia 3 - 4 New Zealand → New Zealand


In [99]:
#Round of 16
r16_matchups = {
    89: (r16_teams[74], r16_teams[77]),
    90: (r16_teams[73], r16_teams[75]),
    91: (r16_teams[76], r16_teams[78]),
    92: (r16_teams[79], r16_teams[80]),
    93: (r16_teams[83], r16_teams[84]),
    94: (r16_teams[81], r16_teams[82]),
    95: (r16_teams[86], r16_teams[88]),
    96: (r16_teams[85], r16_teams[87])
}

r16_rows = []
r8_teams = {}

for match, teams in r16_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]

    r8_teams[match] = winner
    r16_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[teams[0]], 
                                    country_elo[teams[1]])
    
    country_elo[teams[0]] = new_home
    country_elo[teams[1]] = new_away


South Africa 0 - 0 Norway → South Africa
South Korea 2 - 3 Netherlands → Netherlands
Brazil 0 - 1 Germany → Germany
Mexico 4 - 3 Turkey → Mexico
Croatia 0 - 0 Spain → Croatia
Japan 3 - 0 Belgium → Japan
Uruguay 4 - 0 New Zealand → Uruguay
Bosnia and Herzegovina 1 - 1 Panama → Panama


In [100]:
#Quarter final
QF_matchups = {
  97: (r8_teams[89], r8_teams[90]),
  98: (r8_teams[93], r8_teams[94]), 
  99: (r8_teams[91], r8_teams[92]), 
  100: (r8_teams[95], r8_teams[96]), 
}

QF_rows = []
SF_teams = {}

for match, teams in QF_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]

    SF_teams[match] = winner
    QF_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[teams[0]], 
                                    country_elo[teams[1]])
    
    country_elo[teams[0]] = new_home
    country_elo[teams[1]] = new_away


South Africa 3 - 4 Netherlands → Netherlands
Croatia 1 - 1 Japan → Japan
Germany 0 - 0 Mexico → Mexico
Uruguay 3 - 4 Panama → Panama


In [101]:
#Semi-Final

SF_matchups = {
  101: (SF_teams[97], SF_teams[98]),
  102: (SF_teams[99], SF_teams[100]) 
}

SF_rows = []
FINAL_teams = {}

for match, teams in SF_matchups.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]

    FINAL_teams[match] = winner
    SF_rows.append({
        'home_team': teams[0],
        'away_team': teams[1],
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[teams[0]], 
                                    country_elo[teams[1]])
    
    country_elo[teams[0]] = new_home
    country_elo[teams[1]] = new_away

Netherlands 0 - 0 Japan → Netherlands
Mexico 1 - 1 Panama → Panama


In [102]:
for match, teams in {103: (FINAL_teams[101], FINAL_teams[102])}.items():
    #teams[0] = home team, team[1] = away team
    x=predict_match(teams[0], teams[1], wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler, squad_values, country_elo[teams[0]], country_elo[teams[1]])

    if x.outcome == "Home Win": 
        S, winner = 1, teams[0]
    elif x.outcome == "Away Win": 
        S, winner = 0, teams[1]
    else: 
        S = .5
        p_home_wins_pens = 0.5 + (country_elo[teams[0]] - country_elo[teams[1]]) / 4000
        winner = teams[0] if np.random.random() < np.clip(p_home_wins_pens, 0.3, 0.7) else teams[1]
        
    print(f"{teams[0]} {x.home_score} - {x.away_score} {teams[1]} → {winner}")


Netherlands 4 - 0 Panama → Netherlands


In [103]:
results = []
for i in range(1000):
    w = run_tournament(wc_model, draw_threshold, history_dict, h2h_dict,
                       country_elo, model_h, model_a, scaler, group_stage_matches)
    results.append(w)

for team, count in Counter(results).most_common(10):
    print(f"{team}: {count/10:.1f}%")

KeyboardInterrupt: 